# Timetable routing lab: RAPTOR and Connection Scan

This lab compares two timetable-based earliest-arrival baselines. RAPTOR organises results by vehicle boardings; CSA scans connections in departure order. Times are minutes after midnight. These tiny fixtures are deliberately transparent and are not GTFS feeds.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'implementations' / 'python').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations' / 'python' / 'raptor'))
sys.path.insert(0, str(ROOT / 'implementations' / 'python' / 'csa'))
from raptor import Route, Trip, raptor
from csa import Connection, earliest_arrival


## RAPTOR: rounds represent vehicle boardings

A direct trip is found in round 1. A second vehicle requires round 2. A walking transfer can be chained after a transit round.


In [ ]:
r1 = Route(('A', 'B'), (Trip((0, 10), (0, 10)),))
r2 = Route(('B', 'C'), (Trip((15, 25), (15, 25)),))
result = raptor([r1, r2], {'A': 0}, max_transfers=1)
[(round_no, labels.get('C')) for round_no, labels in enumerate(result['rounds'])]


In [ ]:
assert 'C' not in result['rounds'][1]
assert result['rounds'][2]['C'] == 25
walk_result = raptor([r1], {'A': 0}, max_transfers=0,
                     footpaths={'B': [('C', 4)], 'C': [('D', 2)]})
assert walk_result['rounds'][1]['D'] == 16
walk_result['rounds'][1]


## Connection Scan: process connections chronologically

CSA keeps earliest arrival labels and remembers which trips have already been boarded. Minimum transfer time applies when changing trips, but not when continuing on the same trip.


In [ ]:
connections = [
    Connection('T1', 'A', 'B', 5, 10),
    Connection('T1', 'B', 'C', 10, 15),
    Connection('T2', 'C', 'D', 18, 24),
]
scan = earliest_arrival(connections, {'A': 0}, minimum_transfer={'C': 3})
scan['arrival_times']


In [ ]:
assert scan['arrival_times']['C'] == 15
assert scan['arrival_times']['D'] == 24
blocked = earliest_arrival(
    [Connection('T1', 'A', 'B', 5, 10), Connection('T2', 'B', 'C', 12, 20)],
    {'A': 0}, minimum_transfer={'B': 3})
assert 'C' not in blocked['arrival_times']
'All timetable-routing checks passed.'


## Interpretation and limits

RAPTOR's round index makes the number of boardings explicit. CSA is compact when a timetable is already represented as a departure-sorted connection list. Both examples assume a common service-day clock and simplified trip semantics. Neither notebook fixture covers GTFS calendars, frequency-based services, real-time updates, fares, wheelchair constraints or production feed indexing. Compare results only after ensuring both algorithms receive the same transfer and walking assumptions.
